# LLM confabulation / choice blindness: Kaggle runner

Settings: **Accelerator = GPU T4 x2**, **Internet = On**.

1. Generate transcripts for three small open chat models (M0 natural, M1 transcript swap, M2 activation steering, M3 hidden cue).
2. Label every explanation with a blind judge (Qwen2.5-7B-Instruct, split across both GPUs).
3. Summaries, separability tests and figures.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /kaggle/working
!rm -rf llm-confabulation
!git clone --depth 1 https://github.com/pranavsaikosaraju/llm-confabulation.git
%cd llm-confabulation
!pip -q install -U "transformers>=4.56" accelerate 2>&1 | tail -n 2

## 0. Smoke test (tiny model, 40 items per task; about 2 minutes)

In [ ]:
!python scripts/run_p2.py --model Qwen/Qwen2.5-0.5B-Instruct --out results/_smoke --n-per-task 40 --alphas 1,4
!python scripts/judge_p2.py --transcripts results/_smoke/transcripts.csv --judge Qwen/Qwen2.5-0.5B-Instruct
!python scripts/analyze_p2.py --run results/_smoke
!rm -rf results/_smoke

## 1. Transcripts

In [ ]:
!python scripts/run_p2.py --model Qwen/Qwen2.5-1.5B-Instruct --out results/qwen2.5-1.5b-instruct

In [ ]:
!python scripts/run_p2.py --model Qwen/Qwen2.5-3B-Instruct --out results/qwen2.5-3b-instruct

In [ ]:
!python scripts/run_p2.py --model HuggingFaceTB/SmolLM2-1.7B-Instruct --out results/smollm2-1.7b-instruct

## 2. Blind judge

In [ ]:
!python scripts/judge_p2.py --transcripts results/qwen2.5-1.5b-instruct/transcripts.csv --judge Qwen/Qwen2.5-7B-Instruct

In [ ]:
!python scripts/judge_p2.py --transcripts results/qwen2.5-3b-instruct/transcripts.csv --judge Qwen/Qwen2.5-7B-Instruct

In [ ]:
!python scripts/judge_p2.py --transcripts results/smollm2-1.7b-instruct/transcripts.csv --judge Qwen/Qwen2.5-7B-Instruct

## 3. Analysis and hand-label sample

In [ ]:
!python scripts/analyze_p2.py --run results/qwen2.5-1.5b-instruct
!python scripts/hand_labels.py export --run results/qwen2.5-1.5b-instruct

In [ ]:
!python scripts/analyze_p2.py --run results/qwen2.5-3b-instruct
!python scripts/hand_labels.py export --run results/qwen2.5-3b-instruct

In [ ]:
!python scripts/analyze_p2.py --run results/smollm2-1.7b-instruct
!python scripts/hand_labels.py export --run results/smollm2-1.7b-instruct

In [ ]:
!cd results && zip -qr /kaggle/working/p2_results.zip . && ls -la /kaggle/working/p2_results.zip